# 16s Metagenome analysis pipeline
Reaquired file: 1.final.tx.1.cons.taxonomy(Got from mothur analysis)
                2.final.tx.shared

                                                                                                        

# Rename the file "final.tx.1.cons.taxonomy"

In [2]:
mv final.tx.1.cons.taxonomy final.tx.1.cons.taxonomy.NEW

: 1

# Find the "unknown" otu in "final.tx.1.cons.taxonomy" file 
Note: note down the unknown Otu ID same ID u have to remove from "final.tx.shared.NEW.transposed" file 

In [3]:
grep "unknown" final.tx.1.cons.taxonomy.NEW | head -n1 | awk '{print $1;}'

Otu0176l


: 1

# Remove the Otu and rename the file

In [4]:
grep -v "Otu0176" final.tx.1.cons.taxonomy.NEW > final.tx.1.cons.taxonomy.NEW1
rm final.tx.1.cons.taxonomy.NEW
mv final.tx.1.cons.taxonomy.NEW1 final.tx.1.cons.taxonomy.NEW

: 1

# Alpha diversity analysis

In [3]:
require(phyloseq)
require(ggplot2)
require(pheatmap)
library(ape)
library(dplyr)
library(ggplot2)
library(gplots)
library(lme4)
library(phangorn)
library(phyloseq)
library(plotly)
library(tidyr)
library(VennDiagram)
library(venneuler)
require(vegan)
require(reshape)

Loading required package: phyloseq

Loading required package: ggplot2

Loading required package: pheatmap


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘gplots’


The following object is masked from ‘package:stats’:

    lowess


Loading required package: Matrix



ERROR: Error in library(plotly): there is no package called ‘plotly’


In [5]:
install.packages("plotly")

Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done



# Load the final.tx.shared file convert into table form

In [20]:
shared=read.delim("final.tx.shared",header = T,stringsAsFactors = F, sep = "\t")
shared_new=data.frame(t(shared))
write.table(shared_new, file = "final.tx.shared.NEW.transposed",sep = "\t",row.names = T,col.names = F,quote = F)
rm(shared_new)

# Remove same unknown Otu Id from "final.tx.shared.NEW.transposed" file

In [ ]:
grep -v "Otu0118" final.tx.shared.NEW.transposed > final.tx.shared.NEW.transposed1
rm final.tx.shared.NEW.transposed
mv final.tx.shared.NEW.transposed1 final.tx.shared.NEW.transposed

# Convert final.tx.shared.NEW.transposed to final.tx.shared.NEW in table form

In [21]:
shared=read.delim("final.tx.shared.NEW.transposed",header = T,stringsAsFactors = F, sep = "\t")
shared_new=data.frame(t(shared))
write.table(shared_new, file = "final.tx.shared.NEW",sep = "\t",row.names = F,col.names = F,quote = F)
rm(shared_new)

# Load the taxonomy file

In [22]:
import pandas as pd
df=pd.read_csv("final.tx.1.cons.taxonomy.NEW",sep="\t")
df.head()

,OTU,Size,Taxonomy
0,Otu0001,23241,k__Bacteria(100);k__Bacteria_unclassified(100)...
1,Otu0002,22313,k__Bacteria(100);p__Proteobacteria(100);c__Gam...
2,Otu0003,17058,k__Bacteria(100);p__Proteobacteria(100);p__Pro...
3,Otu0004,35098,k__Bacteria(100);p__Bacteroidetes(100);c__Flav...
4,Otu0005,28262,k__Bacteria(100);p__Proteobacteria(100);c__Alp...


# Load the sample file having all Otu's values
convert first row into header, remove otu numberotu's and group colum from table,rename Group colum to OTU colum that will help while merging dataframe 

In [23]:
df1=pd.read_csv("final.tx.shared.NEW.transposed",sep="\t")
df1.head()
df1.columns = df1.iloc[0]
df1.drop(df1.index[:2], inplace=True)
df1=df1.rename(columns={"Group":"OTU"})
df1.head()

,OTU,GR-6,HR-6,KR-1,MR-6,HDR-6,KR-6,VC-1,VS-6
2,Otu0001,382,1369,1646,6645,2323,4551,4334,1991
3,Otu0002,1002,4647,3696,3257,1410,3320,3587,1394
4,Otu0003,1224,2407,1999,3959,776,2652,2561,1480
5,Otu0004,1860,7414,1437,9924,3652,3196,1708,5907
6,Otu0005,4642,4403,2882,5109,636,4158,2909,3523


# Merge Taxonomy file and sample otu's into new dataframe, on the basis of same OTU Id's

In [24]:
df2=pd.merge(df,df1,on="OTU",how="inner")
df2.head()

,OTU,Size,Taxonomy,GR-6,HR-6,KR-1,MR-6,HDR-6,KR-6,VC-1,VS-6
0,Otu0001,23241,k__Bacteria(100);k__Bacteria_unclassified(100)...,382,1369,1646,6645,2323,4551,4334,1991
1,Otu0002,22313,k__Bacteria(100);p__Proteobacteria(100);c__Gam...,1002,4647,3696,3257,1410,3320,3587,1394
2,Otu0003,17058,k__Bacteria(100);p__Proteobacteria(100);p__Pro...,1224,2407,1999,3959,776,2652,2561,1480
3,Otu0004,35098,k__Bacteria(100);p__Bacteroidetes(100);c__Flav...,1860,7414,1437,9924,3652,3196,1708,5907
4,Otu0005,28262,k__Bacteria(100);p__Proteobacteria(100);c__Alp...,4642,4403,2882,5109,636,4158,2909,3523


# Store the _mothur__OTU__Table.tab

In [25]:
df2.to_csv('CG_DN_803_mothur_OTU_Table.tab', sep="\t",index=False)

# Normalized the OTUs on the basis of all samples

In [26]:
otus=read.delim(file = "CG_DN_803_mothur_OTU_Table.tab",header = T,sep = "\t",comment.char = "#",stringsAsFactors = F)
as.matrix(names(otus))
otus_norm=otus
for( i in 4:length(otus_norm))
{
  otus_norm[,i]=round((otus_norm[,i]/sum(otus_norm[,i]))*100000,digits = 1)
}
write.table(otus_norm, file = "CG_DN_803_mothur_OTU_Table.normalized.tab",sep = "\t",row.names = F,col.names = T,quote = F)


OTU
Size
Taxonomy
GR.6
HR.6
KR.1
MR.6
HDR.6
KR.6
VC.1
VS.6


# Relative abundance 

In [27]:
otus=read.delim(file = "CG_DN_803_mothur_OTU_Table.tab",header = T,sep = "\t",comment.char = "#",stringsAsFactors = F)
as.matrix(names(otus))
otus_norm=otus
for( i in 4:length(otus_norm))
{
  otus_norm[,i]=round((otus_norm[,i]/sum(otus_norm[,i]))*100,digits = 2)
}
write.table(otus_norm, file = "CG_DN_803_mothur_OTU_Table.relative-abundance.tab",sep = "\t",row.names = F,col.names = T,quote = F)


OTU
Size
Taxonomy
GR.6
HR.6
KR.1
MR.6
HDR.6
KR.6
VC.1
VS.6


In [28]:
##relative abundance for LEfSe input file
otus=read.delim(file = "CG_DN_803_mothur_OTU_Table.tab",header = T,sep = "\t",comment.char = "#",stringsAsFactors = F)
as.matrix(names(otus))
otus_norm=otus
otus_norm$Taxonomy=gsub("\\(100)", "", otus_norm$Taxonomy)
otus_norm$Taxonomy=gsub("\\;", "|", otus_norm$Taxonomy)
otus_norm$Taxonomy=gsub("k__", "", otus_norm$Taxonomy)
otus_norm$Taxonomy=gsub("\\p__", "", otus_norm$Taxonomy)
otus_norm$Taxonomy=gsub("\\c__", "", otus_norm$Taxonomy)
otus_norm$Taxonomy=gsub("\\o__", "", otus_norm$Taxonomy)
otus_norm$Taxonomy=gsub("f__", "", otus_norm$Taxonomy)
otus_norm$Taxonomy=gsub("\\g__", "", otus_norm$Taxonomy)
otus_norm$Taxonomy=gsub("s__", "", otus_norm$Taxonomy)

for( i in 4:length(otus_norm))
{
  otus_norm[,i]=round((otus_norm[,i]/sum(otus_norm[,i]))*100,digits = 2)
}
write.table(otus_norm, file = "CG_DN_803_mothur_OTU_Table.relative-abundance.LEfSe-file.tab",sep = "\t",row.names = F,col.names = T,quote = F)


OTU
Size
Taxonomy
GR.6
HR.6
KR.1
MR.6
HDR.6
KR.6
VC.1
VS.6


In [29]:
#####Normalized OTUs
#Input for STAMP Analysis
otus=read.delim(file = "CG_DN_803_mothur_OTU_Table.tab",header = T,sep = "\t",comment.char = "#",stringsAsFactors = F)
as.matrix(names(otus))
otus_norm=otus
for( i in 4:length(otus_norm))
{
  otus_norm[,i]=round((otus_norm[,i]/sum(otus_norm[,i]))*100000,digits = 1)
}

normalized_OTUs=otus_norm
rownames(normalized_OTUs)=normalized_OTUs$OTU
#For STAMP input
as.matrix(names(normalized_OTUs))
normalized_OTUs1=normalized_OTUs[,c(3:ncol(normalized_OTUs))]
colnames(normalized_OTUs1)=sub(pattern = "Taxonomy",replacement = "Feature", colnames(normalized_OTUs1))
normalized_OTUs1$Feature=gsub("\\(100)", "", normalized_OTUs1$Feature)
write.table(normalized_OTUs1, file = "Normalized.OTU.spf",sep = "\t",row.names = F,col.names = T,quote = F)

OTU
Size
Taxonomy
GR.6
HR.6
KR.1
MR.6
HDR.6
KR.6
VC.1
VS.6


OTU
Size
Taxonomy
GR.6
HR.6
KR.1
MR.6
HDR.6
KR.6
VC.1
VS.6


# Splitting OTU_table file taxonomy column into all features
and rename the column with there specific name

In [30]:
df3=df2.Taxonomy.apply(lambda x: pd.Series(str(x).split(";")))
df3=df3.rename(columns={0:"Kingdom",1:"Phylum",2:"Class",3:"Order",4:"Family",5:"Genus",6:"Species"})
df3.head()

,Kingdom,Phylum,Class,Order,Family,Genus,Species,7
0,k__Bacteria(100),k__Bacteria_unclassified(100),k__Bacteria_unclassified(100),k__Bacteria_unclassified(100),k__Bacteria_unclassified(100),k__Bacteria_unclassified(100),k__Bacteria_unclassified(100),
1,k__Bacteria(100),p__Proteobacteria(100),c__Gammaproteobacteria(100),c__Gammaproteobacteria_unclassified(100),c__Gammaproteobacteria_unclassified(100),c__Gammaproteobacteria_unclassified(100),c__Gammaproteobacteria_unclassified(100),
2,k__Bacteria(100),p__Proteobacteria(100),p__Proteobacteria_unclassified(100),p__Proteobacteria_unclassified(100),p__Proteobacteria_unclassified(100),p__Proteobacteria_unclassified(100),p__Proteobacteria_unclassified(100),
3,k__Bacteria(100),p__Bacteroidetes(100),c__Flavobacteriia(100),o__Flavobacteriales(100),f__Flavobacteriaceae(100),f__Flavobacteriaceae_unclassified(100),f__Flavobacteriaceae_unclassified(100),
4,k__Bacteria(100),p__Proteobacteria(100),c__Alphaproteobacteria(100),c__Alphaproteobacteria_unclassified(100),c__Alphaproteobacteria_unclassified(100),c__Alphaproteobacteria_unclassified(100),c__Alphaproteobacteria_unclassified(100),


# Drop the last column having NaN value, concatenate the splitted taxonomy column with OTU_table file and drop the taxonomy column

In [31]:
df3=df3.drop([7],axis=1)
result = pd.concat([df2, df3], axis=1, sort=False)
df4=result.drop(['Taxonomy'],axis=1)

In [32]:
df4.columns

Index(['OTU', 'Size', 'GR-6', 'HR-6', 'KR-1', 'MR-6', 'HDR-6', 'KR-6', 'VC-1',
       'VS-6', 'Kingdom', 'Phylum', 'Class', 'Order', 'Family', 'Genus',
       'Species'],
      dtype='object')

In [33]:
df5=df4[['OTU', 'Size', 'Kingdom', 'Phylum', 'Class', 'Order', 'Family', 'Genus',
       'Species','GR-6', 'HR-6', 'KR-1', 'MR-6', 'HDR-6', 'KR-6', 'VC-1',
       'VS-6']]

In [34]:
df5['Kingdom'] = df5['Kingdom'].map(lambda x: x.lstrip('k__').rstrip('k__'))
df5['Phylum'] = df5['Phylum'].map(lambda x: x.lstrip('k__').rstrip('k__'))
df5['Phylum'] = df5['Phylum'].map(lambda x: x.lstrip('p__').rstrip('p__'))

df5['Class'] = df5['Class'].map(lambda x: x.lstrip('k__').rstrip('k__'))
df5['Class'] = df5['Class'].map(lambda x: x.lstrip('c__').rstrip('c__'))
df5['Class'] = df5['Class'].map(lambda x: x.lstrip('p__').rstrip('p__'))

df5['Order'] = df5['Order'].map(lambda x: x.lstrip('k__').rstrip('k__'))
df5['Order'] = df5['Order'].map(lambda x: x.lstrip('o__').rstrip('o__'))
df5['Order'] = df5['Order'].map(lambda x: x.lstrip('p__').rstrip('p__'))
df5['Order'] = df5['Order'].map(lambda x: x.lstrip('c__').rstrip('c__'))

df5['Family'] = df5['Family'].map(lambda x: x.lstrip('k__').rstrip('k__'))
df5['Family'] = df5['Family'].map(lambda x: x.lstrip('o__').rstrip('o__'))
df5['Family'] = df5['Family'].map(lambda x: x.lstrip('p__').rstrip('p__'))
df5['Family'] = df5['Family'].map(lambda x: x.lstrip('c__').rstrip('c__'))
df5['Family'] = df5['Family'].map(lambda x: x.lstrip('f__').rstrip('f__'))

df5['Genus'] = df5['Genus'].map(lambda x: x.lstrip('k__').rstrip('k__'))
df5['Genus'] = df5['Genus'].map(lambda x: x.lstrip('o__').rstrip('o__'))
df5['Genus'] = df5['Genus'].map(lambda x: x.lstrip('p__').rstrip('p__'))
df5['Genus'] = df5['Genus'].map(lambda x: x.lstrip('c__').rstrip('c__'))
df5['Genus'] = df5['Genus'].map(lambda x: x.lstrip('f__').rstrip('f__'))
df5['Genus'] = df5['Genus'].map(lambda x: x.lstrip('g__').rstrip('g__'))

df5['Species'] = df5['Species'].map(lambda x: x.lstrip('k__').rstrip('k__'))
df5['Species'] = df5['Species'].map(lambda x: x.lstrip('o__').rstrip('o__'))
df5['Species'] = df5['Species'].map(lambda x: x.lstrip('p__').rstrip('p__'))
df5['Species'] = df5['Family'].map(lambda x: x.lstrip('c__').rstrip('c__'))
df5['Species'] = df5['Species'].map(lambda x: x.lstrip('f__').rstrip('f__'))
df5['Species'] = df5['Species'].map(lambda x: x.lstrip('g__').rstrip('g__'))


In [35]:
df5['Species'] = df5['Species'].map(lambda x: x.lstrip('(100)').rstrip('(100)'))
df5['Kingdom'] = df5['Kingdom'].map(lambda x: x.lstrip('(100)').rstrip('(100)'))
df5['Phylum'] = df5['Phylum'].map(lambda x: x.lstrip('(100)').rstrip('(100)'))
df5['Class'] = df5['Class'].map(lambda x: x.lstrip('(100)').rstrip('(100)'))
df5['Order'] = df5['Order'].map(lambda x: x.lstrip('(100)').rstrip('(100)'))
df5['Family'] = df5['Family'].map(lambda x: x.lstrip('(100)').rstrip('(100)'))
df5['Genus'] = df5['Genus'].map(lambda x: x.lstrip('(100)').rstrip('(100)'))

In [36]:
df5.head()

,OTU,Size,Kingdom,Phylum,Class,Order,Family,Genus,Species,GR-6,HR-6,KR-1,MR-6,HDR-6,KR-6,VC-1,VS-6
0,Otu0001,23241,Bacteria,Bacteria_unclassified,Bacteria_unclassified,Bacteria_unclassified,Bacteria_unclassified,Bacteria_unclassified,Bacteria_unclassified,382,1369,1646,6645,2323,4551,4334,1991
1,Otu0002,22313,Bacteria,Proteobacteria,Gammaproteobacteria,Gammaproteobacteria_unclassified,Gammaproteobacteria_unclassified,Gammaproteobacteria_unclassified,Gammaproteobacteria_unclassified,1002,4647,3696,3257,1410,3320,3587,1394
2,Otu0003,17058,Bacteria,Proteobacteria,Proteobacteria_unclassified,Proteobacteria_unclassified,Proteobacteria_unclassified,Proteobacteria_unclassified,Proteobacteria_unclassified,1224,2407,1999,3959,776,2652,2561,1480
3,Otu0004,35098,Bacteria,Bacteroidetes,Flavobacteriia,Flavobacteriales,Flavobacteriaceae,Flavobacteriaceae_unclassified,Flavobacteriaceae,1860,7414,1437,9924,3652,3196,1708,5907
4,Otu0005,28262,Bacteria,Proteobacteria,Alphaproteobacteria,Alphaproteobacteria_unclassified,Alphaproteobacteria_unclassified,Alphaproteobacteria_unclassified,Alphaproteobacteria_unclassified,4642,4403,2882,5109,636,4158,2909,3523


# Save the file as OTU_table.splitted

In [37]:
df5.to_csv("CG_DN_803_mothur_OTU_Table.splitted.tab",sep="\t",index=False)

In [42]:
mkdir Krona_pie-chart

: 1

# Extract all taxonomy features and saving into krona file format to visualize in pie-chart form

In [43]:
###make Krona input files
rawOTUs=read.delim(file = "CG_DN_803_mothur_OTU_Table.splitted.tab",header = T,sep = "\t",comment.char = "#",stringsAsFactors = F)
as.matrix(names(rawOTUs))
column_names=colnames(rawOTUs)
for( i in 9:length(rawOTUs))
{
  filename=paste("Krona_pie-chart/",column_names[i],"_krona.in",sep = "")
  write.table(rawOTUs[,c(i,3:9)], file = filename,sep = "\t",row.names = F,col.names = F,quote = F)
}
rm(rawOTUs)

OTU
Size
Kingdom
Phylum
Class
Order
Family
Genus
Species
GR.6
HR.6


# Perl script to get the hmtl file

In [54]:
cd Krona_pie-chart
perl /home/clevergene/Downloads/KronaTools-2.8/scripts/ImportText.pl *in
cd ..

Writing text.krona.html...


: 1

In [45]:
new=pd.read_csv("final.tx.shared",sep="\t")
new.head()

,label,Group,numOtus,Otu0001,Otu0002,Otu0003,Otu0004,Otu0005,Otu0006,Otu0007,...,Otu1488,Otu1489,Otu1490,Otu1491,Otu1492,Otu1493,Otu1494,Otu1495,Otu1496,Otu1497
0,1,GR-6,1497,382,1002,1224,1860,4642,4827,7551,...,0,0,0,0,0,0,0,0,0,0
1,1,HR-6,1497,1369,4647,2407,7414,4403,294,6133,...,0,0,0,0,0,0,0,0,0,0
2,1,KR-1,1497,1646,3696,1999,1437,2882,146,2340,...,1,1,1,0,0,0,0,0,0,0
3,1,MR-6,1497,6645,3257,3959,9924,5109,3639,16940,...,0,0,0,0,1,0,0,0,0,0
4,1,HDR-6,1497,2323,1410,776,3652,636,151,1126,...,0,0,0,0,0,1,0,0,0,1


In [46]:
new1=pd.read_csv("final.tx.shared.NEW",sep='\t')
new1.head()

,Group,numOtus,Otu0001,Otu0002,Otu0003,Otu0004,Otu0005,Otu0006,Otu0007,Otu0008,...,Otu1488,Otu1489,Otu1490,Otu1491,Otu1492,Otu1493,Otu1494,Otu1495,Otu1496,Otu1497
0,GR-6,1497,382,1002,1224,1860,4642,4827,7551,1437,...,0,0,0,0,0,0,0,0,0,0
1,HR-6,1497,1369,4647,2407,7414,4403,294,6133,4895,...,0,0,0,0,0,0,0,0,0,0
2,KR-1,1497,1646,3696,1999,1437,2882,146,2340,1980,...,1,1,1,0,0,0,0,0,0,0
3,MR-6,1497,6645,3257,3959,9924,5109,3639,16940,4013,...,0,0,0,0,1,0,0,0,0,0
4,HDR-6,1497,2323,1410,776,3652,636,151,1126,5259,...,0,0,0,0,0,1,0,0,0,1


In [47]:
new_label=new[['label']]
final = pd.concat([new, new_label], axis=1, sort=False)
final.to_csv("final.tx.shared.NEW",index=False,sep="\t")

# ############ ALPHA DIVERSITY #######################

#shared file

In [50]:
APS_mothur=import_mothur(mothur_shared_file = "final.tx.shared.NEW",mothur_constaxonomy_file = "final.tx.1.cons.taxonomy.NEW")
colnames(tax_table(APS_mothur))=c("Kingdom","Phylum", "Class", "Order", "Family", "Genus","species")
sampleID=colnames(APS_mothur@otu_table)
sampleID

[1] "GR-6"  "HR-6"  "KR-1"  "MR-6"  "HDR-6" "KR-6"  "VC-1"  "VS-6"

KeyError: '[7] not found in axis'

In [51]:
sampleType=c( 'GR-6','HR-6','KR-1','MR-6','HDR-6','KR-6','VC-1','VS-6')
sampletable=data.frame(cbind(sampleID,sampleType),stringsAsFactors = FALSE)
colnames(sampletable)=c("SampleID","SampleType")
rownames(sampletable)=sampleID
sample_data(APS_mothur)=sampletable


# PCoA PLOT
#PCoA PLOT New with OTUs samples

In [52]:

png("PCoA_plot.png", type="cairo", res = 250, units="in", width=7, height=6)
APS_mothur_ordinates_pca=ordinate(APS_mothur,"PCoA")
#PCA=plot_ordination(physeq = APS_mothur,ordination = res_mothur_ordinates_pca,type = "samples",shape = "SampleType",color = "SampleType",title = "PCoA")
PCA=plot_ordination(physeq = APS_mothur,ordination = APS_mothur_ordinates_pca,type = "samples",shape = "SampleType",color = "SampleType",title = "PCoA",label = "SampleID")
PCA +geom_point(size=1.5)
dev.off()

Warning message:
“The shape palette can deal with a maximum of 6 discrete values because
more than 6 becomes difficult to discriminate; you have 8. Consider
specifying shapes manually if you must have them.”
Warning message:
“Removed 2 rows containing missing values (geom_point).”


png 
  2

In [6]:
####################################################
############ ALPHA DIVERSITY #######################
####################################################
#shared file
APS_mothur=import_mothur(mothur_shared_file = "final.tx.shared.NEW",mothur_constaxonomy_file = "final.tx.1.cons.taxonomy.NEW")
colnames(tax_table(APS_mothur))=c("Kingdom","Phylum", "Class", "Order", "Family", "Genus","species")
sampleID=colnames(APS_mothur@otu_table)
#sampleType=c("Sample_2669_18A" , "Sample_2669_18B" , "Sample_2670_18A" , "Sample_2670_18B" , "Sample_2672_18A" , "Sample_2672_18B" , "Sample_2673_18A" , "Sample_2673_18B" , "Sample_2984_18A" , "Sample_2984_18B" , "Sample_3773_18A" , "Sample_3773_18B" , "Sample_3776_18A" , "Sample_3776_18B" , "Sample_3777_18A" , "Sample_3777_18B" , "Sample_3778_18A" , "Sample_3778_18B" , "Sample_3779_18A" , "Sample_3779_18B")
sampleType=c('GR-6', 'HR-6', 'KR-1', 'MR-6', 'HDR-6', 'KR-6', 'VC-1','VS-6')
sampletable=data.frame(cbind(sampleID,sampleType),stringsAsFactors = FALSE)
colnames(sampletable)=c("SampleID","SampleType")
rownames(sampletable)=sampleID
sample_data(APS_mothur)=sampletable

#PCoA PLOT
#PCoA PLOT New without OTUs only samples
png("PCoA_plot.png", type="cairo", res = 250, units="in", width=7, height=6)
APS_mothur_ordinates_pca=ordinate(APS_mothur,"PCoA")
#PCA=plot_ordination(physeq = APS_mothur,ordination = res_mothur_ordinates_pca,type = "samples",shape = "SampleType",color = "SampleType",title = "PCoA")
PCA=plot_ordination(physeq = APS_mothur,ordination = APS_mothur_ordinates_pca,type = "samples",shape = "SampleType",color = "SampleType",title = "PCoA",label = "SampleID")
PCA +geom_point(size=1.5)
dev.off()


#### AlphaDiversity plot####
png("AlphaDiversity_plot.Group-wise.png", type="cairo", res = 250, units="in", width=15, height=10)
AlphaDiversity=plot_richness(APS_mothur,color = "SampleType",x="SampleType")
AlphaDiversity +theme(axis.text.x = element_text(size = 6.5, angle = 90, hjust = 1))
dev.off()

png("AlphaDiversity_plot.Chao1_ACE_Shannon.sample-wise.png", type="cairo", res = 250, units="in", width=14, height=10)
AlphaDiversity=plot_richness(APS_mothur,color = "SampleType",measures=c("Chao1", "ACE", "Shannon"))
AlphaDiversity +theme(axis.text.x = element_text(size = 7, angle = 90, hjust = 1))
dev.off()
 
png("AlphaDiversity_plot.Simpson_InvSimpson_Fisher.sample-wise.png", type="cairo", res = 250, units="in", width=12, height=10)
AlphaDiversity=plot_richness(APS_mothur,color = "SampleType",measures=c("Simpson", "InvSimpson", "Fisher"))
AlphaDiversity +theme(axis.text.x = element_text(size = 5, angle = 90, hjust = 1))
dev.off()

#table
AlphaDiversity_table=estimate_richness(APS_mothur)
write.table(AlphaDiversity_table, file = "AlphaDiversity_table.txt",sep = "\t",row.names = T,col.names = T,quote = F)

### NEW rarefraction plot
library(vegan)
png("Rarefraction_plot.png", type="cairo", res = 200, units="in", width=14, height=10)
rarecurve(t(otu_table(APS_mothur)))
dev.off()



Warning message:
“The shape palette can deal with a maximum of 6 discrete values because
more than 6 becomes difficult to discriminate; you have 8. Consider
specifying shapes manually if you must have them.”
Warning message:
“Removed 2 rows containing missing values (geom_point).”


png 
  2

png 
  2

png 
  2

png 
  2

Loading required package: permute

Loading required package: lattice

This is vegan 2.5-7


Attaching package: ‘vegan’


The following objects are masked from ‘package:phangorn’:

    diversity, treedist




png 
  2

In [58]:
### NEW rarefraction plot
library(vegan)
png("Rarefraction_plot.png", type="cairo", res = 200, units="in", width=14, height=10)
rarecurve(t(otu_table(APS_mothur)))
dev.off()

png 
  2

In [ ]:

##############################################
###table,barplot and heatmap phylum level#####
##############################################
#Table
APS_mothur_phylum=tax_glom(APS_mothur,taxrank = "Phylum")
#phylum=cbind(data.frame(APS_mothur_phylum@tax_table),data.frame(APS_mothur_phylum@otu_table))
phylum=cbind(APS_mothur_phylum@tax_table,APS_mothur_phylum@otu_table)
phylum=data.frame(phylum,stringsAsFactors = F)
phylum$OTU=rownames(phylum)
as.matrix(names(phylum))
phylum=phylum[,c(ncol(phylum),1,2,7:(ncol(phylum)-1))]
as.matrix(names(phylum))
write.table(phylum, file="Phylum_level.txt", sep = "\t",row.names = F,col.names = T, quote = F)


In [ ]:
#Bar Plot
#APS_mothur_phylum10 = names(sort(taxa_sums(APS_mothur_phylum), TRUE)[1:10]))
APS_mothur_phylum10 = data.frame(sort(taxa_sums(APS_mothur_phylum), TRUE)[1:10])
APS_mothur_phylum10$OTU=rownames(APS_mothur_phylum10)
APS_mothur_phylum10=APS_mothur_phylum10[,c(2,1)]
colnames(APS_mothur_phylum10)=c("OTU","Size")
APS_mothur_phylum10.merge=merge(APS_mothur_phylum10,phylum,by = "OTU",all.x = T)
APS_mothur_phylum10.merge=APS_mothur_phylum10.merge[rev(order(APS_mothur_phylum10.merge$Size)),]
as.matrix(names(APS_mothur_phylum10.merge))
APS_mothur_phylum10.merge=APS_mothur_phylum10.merge[,c(4:ncol(APS_mothur_phylum10.merge))]
#APS_mothur_phylum10.merge$Athul_MetG1 <- as.numeric(as.character(APS_mothur_phylum10.merge$Athul_MetG1))
cols.num <- names(APS_mothur_phylum10.merge[,c(2:ncol(APS_mothur_phylum10.merge))])
APS_mothur_phylum10.merge[cols.num] <- sapply(APS_mothur_phylum10.merge[cols.num],as.numeric)

for( i in 2:length(APS_mothur_phylum10.merge))
{
  APS_mothur_phylum10.merge[,i]=round((APS_mothur_phylum10.merge[,i]/sum(APS_mothur_phylum10.merge[,i]))*100,digits = 1)
}
rm(i)
APS_mothur_phylum10.merge_melt=melt(APS_mothur_phylum10.merge)
colnames(APS_mothur_phylum10.merge_melt)=c("Phylum","Sample","Percentage")
png("Top10_normalized_OTUs_Phylum_Barplot.png", type="cairo", res = 200, units="in", width=10, height=6)
#ggplot(top10phylum_barplot_melt,aes(x = Sample,y = Count,fill=Phylum))+geom_bar(stat = "identity")
phylum_colors=rainbow(length(unique(APS_mothur_phylum10.merge$Phylum)), s=.6, v=.9)[sample(1:length(unique(APS_mothur_phylum10.merge$Phylum)),length(unique(APS_mothur_phylum10.merge$Phylum)))]
ggplot(APS_mothur_phylum10.merge_melt,aes(x = Sample,y = Percentage,fill=Phylum))+
  geom_bar(stat = "identity") + 
  scale_fill_manual(values = phylum_colors)+
  theme(axis.text.x = element_text(angle = 0, hjust = 1))
dev.off()


In [ ]:
#Heatmap
APS_mothur_phylum20 = data.frame(sort(taxa_sums(APS_mothur_phylum), TRUE)[1:20])
APS_mothur_phylum20$OTU=rownames(APS_mothur_phylum20)
APS_mothur_phylum20=APS_mothur_phylum20[,c(2,1)]
colnames(APS_mothur_phylum20)=c("OTU","Size")
APS_mothur_phylum20.merge=merge(APS_mothur_phylum20,phylum,by = "OTU",all.x = T)
APS_mothur_phylum20.merge=APS_mothur_phylum20.merge[rev(order(APS_mothur_phylum20.merge$Size)),]
as.matrix(names(APS_mothur_phylum20.merge))
APS_mothur_phylum20.merge=APS_mothur_phylum20.merge[,c(4:ncol(APS_mothur_phylum20.merge))]
#APS_mothur_phylum20.merge$Athul_MetG1 <- as.numeric(as.character(APS_mothur_phylum20.merge$Athul_MetG1))
cols.num <- names(APS_mothur_phylum20.merge[,c(2:ncol(APS_mothur_phylum20.merge))])
APS_mothur_phylum20.merge[cols.num] <- sapply(APS_mothur_phylum20.merge[cols.num],as.numeric)

for( i in 2:length(APS_mothur_phylum20.merge))
{
  APS_mothur_phylum20.merge[,i]=round((APS_mothur_phylum20.merge[,i]/sum(APS_mothur_phylum20.merge[,i]))*100,digits = 1)
}
rm(i)
rownames(APS_mothur_phylum20.merge)=APS_mothur_phylum20.merge$Phylum
APS_mothur_phylum20.merge=APS_mothur_phylum20.merge[,-1]
# anno=data.frame(factor(c(rep("A",25),rep("B",12),rep("C",6),rep("D",3))))
# rownames(anno)=colnames(APS_mothur_phylum20.merge)
# colnames(anno)="Group"
png("Top20_normalized_OTUs_Phylum_heatmap.png", type="cairo", res = 200, units="in", width=7, height=4)
# pheatmap(APS_mothur_phylum20.merge,show_rownames = T,show_colnames = T,cluster_cols = F,scale = "none",annotation_col = anno,border_color = F,fontsize_col = 7,fontsize_row = 8,fontsize = 7,cellheight = 10)
pheatmap(APS_mothur_phylum20.merge,show_rownames = T,show_colnames = T,cluster_cols = F,scale = "none",border_color = F,fontsize_col = 7,fontsize_row = 8,fontsize = 7,angle_col = 0)
dev.off()



In [ ]:
# ##############################################
# ###table,barplot and heatmap species level#####
# ##############################################
# #Table
# APS_mothur_species=tax_glom(APS_mothur,taxrank = "Species")
# # species=cbind(data.frame(APS_mothur_species@tax_table),data.frame(APS_mothur_species@otu_table))
# species=cbind(APS_mothur_species@tax_table,APS_mothur_species@otu_table)
# species=data.frame(species,stringsAsFactors = F)
# species$OTU=rownames(species)
# as.matrix(names(species))
# species=species[,c(ncol(species),1:(ncol(species)-1))]
# write.table(species, file="Species_level.txt", sep = "\t",row.names = F,col.names = T, quote = F)
# 
# #Bar Plot
# APS_mothur_species10 = data.frame(sort(taxa_sums(APS_mothur_species), TRUE)[1:10])
# APS_mothur_species10$OTU=rownames(APS_mothur_species10)
# APS_mothur_species10=APS_mothur_species10[,c(2,1)]
# colnames(APS_mothur_species10)=c("OTU","Size")
# APS_mothur_species10.merge=merge(APS_mothur_species10,species,by = "OTU",all.x = T)
# APS_mothur_species10.merge=APS_mothur_species10.merge[rev(order(APS_mothur_species10.merge$Size)),]
# as.matrix(names(APS_mothur_species10.merge))
# APS_mothur_species10.merge=APS_mothur_species10.merge[,c(9:ncol(APS_mothur_species10.merge))]
# cols.num <- names(APS_mothur_species10.merge[,c(2:ncol(APS_mothur_species10.merge))])
# APS_mothur_species10.merge[cols.num] <- sapply(APS_mothur_species10.merge[cols.num],as.numeric)
# 
# for( i in 2:length(APS_mothur_species10.merge))
# {
#   APS_mothur_species10.merge[,i]=round((APS_mothur_species10.merge[,i]/sum(APS_mothur_species10.merge[,i]))*100,digits = 1)
# }
# rm(i)
# APS_mothur_species10.merge_melt=melt(APS_mothur_species10.merge)
# colnames(APS_mothur_species10.merge_melt)=c("Species","Sample","Percentage")
# png("Top10_normalized_OTUs_species_Barplot.png", type="cairo", res = 200, units="in", width=9, height=7)
# #ggplot(top10species_barplot_melt,aes(x = Sample,y = Count,fill=species))+geom_bar(stat = "identity")
# species_colors=rainbow(length(unique(APS_mothur_species10.merge$Species)), s=.6, v=.9)[sample(1:length(unique(APS_mothur_species10.merge$Species)),length(unique(APS_mothur_species10.merge$Species)))]
# ggplot(APS_mothur_species10.merge_melt,aes(x = Sample,y = Percentage,fill=Species))+
#   geom_bar(stat = "identity") + 
#   scale_fill_manual(values = species_colors)+
#   theme(axis.text.x = element_text(angle = 0, hjust = 1))
# dev.off()
# 
# #Heatmap
# APS_mothur_species20 = data.frame(sort(taxa_sums(APS_mothur_species), TRUE)[1:20])
# APS_mothur_species20$OTU=rownames(APS_mothur_species20)
# APS_mothur_species20=APS_mothur_species20[,c(2,1)]
# colnames(APS_mothur_species20)=c("OTU","Size")
# APS_mothur_species20.merge=merge(APS_mothur_species20,species,by = "OTU",all.x = T)
# APS_mothur_species20.merge=APS_mothur_species20.merge[rev(order(APS_mothur_species20.merge$Size)),]
# as.matrix(names(APS_mothur_species20.merge))
# APS_mothur_species20.merge=APS_mothur_species20.merge[,c(9:ncol(APS_mothur_species20.merge))]
# cols.num <- names(APS_mothur_species20.merge[,c(2:ncol(APS_mothur_species20.merge))])
# APS_mothur_species20.merge[cols.num] <- sapply(APS_mothur_species20.merge[cols.num],as.numeric)
# 
# for( i in 2:length(APS_mothur_species20.merge))
# {
#   APS_mothur_species20.merge[,i]=round((APS_mothur_species20.merge[,i]/sum(APS_mothur_species20.merge[,i]))*100,digits = 1)
# }
# rm(i)
# rownames(APS_mothur_species20.merge)=APS_mothur_species20.merge$Species
# APS_mothur_species20.merge=APS_mothur_species20.merge[,-1]
# # anno=data.frame(factor(c(rep("A",25),rep("B",12),rep("C",6),rep("D",3))))
# # rownames(anno)=colnames(APS_mothur_species20.merge)
# # colnames(anno)="Group"
# png("Top20_normalized_OTUs_species_heatmap.png", type="cairo", res = 200, units="in", width=6, height=4)
# # pheatmap(APS_mothur_species20.merge,show_rownames = T,show_colnames = T,cluster_cols = F,scale = "none",annotation_col = anno,bspecies_color = F,fontsize_col = 7,fontsize_row = 8,fontsize = 7,cellheight = 10)
# pheatmap(APS_mothur_species20.merge,show_rownames = T,show_colnames = T,cluster_cols = F,scale = "none",border_color = F,fontsize_col = 7,fontsize_row = 8,fontsize = 7,angle_col = 0)
# dev.off()
# 



In [ ]:
##############################################
###table,barplot and heatmap Genus level#####
##############################################
#Table
APS_mothur_Genus=tax_glom(APS_mothur,taxrank = "Genus")
# Genus=cbind(data.frame(APS_mothur_Genus@tax_table),data.frame(APS_mothur_Genus@otu_table))
Genus=cbind(APS_mothur_Genus@tax_table,APS_mothur_Genus@otu_table)
Genus=data.frame(Genus,stringsAsFactors = F)
Genus$OTU=rownames(Genus)
as.matrix(names(Genus))
Genus=Genus[,c(ncol(Genus),1:6,7:(ncol(Genus)-1))]
# colnames(Genus)=c("OTU","Kingdom","Phylum","Class","Order","Family","Genus","Sample-1")
write.table(Genus, file="Genus_level.txt", sep = "\t",row.names = F,col.names = T, quote = F)

#Bar Plot
APS_mothur_Genus10 = data.frame(sort(taxa_sums(APS_mothur_Genus), TRUE)[1:10])
APS_mothur_Genus10$OTU=rownames(APS_mothur_Genus10)
APS_mothur_Genus10=APS_mothur_Genus10[,c(2,1)]
colnames(APS_mothur_Genus10)=c("OTU","Size")
APS_mothur_Genus10.merge=merge(APS_mothur_Genus10,Genus,by = "OTU",all.x = T)
APS_mothur_Genus10.merge=APS_mothur_Genus10.merge[rev(order(APS_mothur_Genus10.merge$Size)),]
as.matrix(names(APS_mothur_Genus10.merge))
APS_mothur_Genus10.merge=APS_mothur_Genus10.merge[,c(7:ncol(APS_mothur_Genus10.merge))]
#APS_mothur_Genus10.merge=APS_mothur_Genus10.merge[,c(6:ncol(APS_mothur_Genus10.merge))]
#APS_mothur_Genus10.merge$Athul_MetG1 <- as.numeric(as.character(APS_mothur_Genus10.merge$Athul_MetG1))
#write.table(APS_mothur_Genus10.merge, file="Genus10.txt", sep = "\t",row.names = F,col.names = T, quote = F)
#APS_mothur_Genus10.merge=read.delim("Genus10.txt",header = T,stringsAsFactors = F, sep = "\t")
as.matrix(names(APS_mothur_Genus10.merge))
#colnames(APS_mothur_Genus10.merge)=c("Genus","Sample-1","Sample-2","Sample-3","Sample-4","Sample-5","Sample-6")
cols.num <- names(APS_mothur_Genus10.merge[,c(2:ncol(APS_mothur_Genus10.merge))])
APS_mothur_Genus10.merge[cols.num] <- sapply(APS_mothur_Genus10.merge[cols.num],as.numeric)

for( i in 2:length(APS_mothur_Genus10.merge))
{
  APS_mothur_Genus10.merge[,i]=round((APS_mothur_Genus10.merge[,i]/sum(APS_mothur_Genus10.merge[,i]))*100,digits = 1)
}
rm(i)
APS_mothur_Genus10.merge_melt=melt(APS_mothur_Genus10.merge)
colnames(APS_mothur_Genus10.merge_melt)=c("Genus","Sample","Percentage")
png("Top10_normalized_OTUs_Genus_Barplot.png", type="cairo", res = 200, units="in", width=10, height=6)
#ggplot(top10Genus_barplot_melt,aes(x = Sample,y = Count,fill=Genus))+geom_bar(stat = "identity")
Genus_colors=rainbow(length(unique(APS_mothur_Genus10.merge$Genus)), s=.6, v=.9)[sample(1:length(unique(APS_mothur_Genus10.merge$Genus)),length(unique(APS_mothur_Genus10.merge$Genus)))]
ggplot(APS_mothur_Genus10.merge_melt,aes(x = Sample,y = Percentage,fill=Genus))+
  geom_bar(stat = "identity") + 
  scale_fill_manual(values = Genus_colors)+
  theme(axis.text.x = element_text(angle = 0, hjust = 1))
dev.off()


In [ ]:
#Heatmap
APS_mothur_Genus20 = data.frame(sort(taxa_sums(APS_mothur_Genus), TRUE)[1:20])
APS_mothur_Genus20$OTU=rownames(APS_mothur_Genus20)
APS_mothur_Genus20=APS_mothur_Genus20[,c(2,1)]
colnames(APS_mothur_Genus20)=c("OTU","Size")
APS_mothur_Genus20.merge=merge(APS_mothur_Genus20,Genus,by = "OTU",all.x = T)
APS_mothur_Genus20.merge=APS_mothur_Genus20.merge[rev(order(APS_mothur_Genus20.merge$Size)),]
as.matrix(names(APS_mothur_Genus20.merge))
APS_mothur_Genus20.merge=APS_mothur_Genus20.merge[,c(7:ncol(APS_mothur_Genus20.merge))]
#APS_mothur_Genus20.merge$`Sample-1` <- as.numeric(as.character(APS_mothur_Genus20.merge$`Sample-1`))
write.table(APS_mothur_Genus20.merge, file="Genus20.txt", sep = "\t",row.names = F,col.names = T, quote = F)
APS_mothur_Genus20.merge=read.delim("Genus20.txt",header = T,stringsAsFactors = F, sep = "\t")
as.matrix(names(APS_mothur_Genus20.merge))
#colnames(APS_mothur_Genus20.merge)=c("Genus","Sample-1","Sample-2","Sample-3","Sample-4","Sample-5","Sample-6")
cols.num <- names(APS_mothur_Genus20.merge[,c(2:ncol(APS_mothur_Genus20.merge))])
APS_mothur_Genus20.merge[cols.num] <- sapply(APS_mothur_Genus20.merge[cols.num],as.numeric)

for( i in 2:length(APS_mothur_Genus20.merge))
{
  APS_mothur_Genus20.merge[,i]=round((APS_mothur_Genus20.merge[,i]/sum(APS_mothur_Genus20.merge[,i]))*100,digits = 1)
}
rm(i)
rownames(APS_mothur_Genus20.merge)=APS_mothur_Genus20.merge$Genus
APS_mothur_Genus20.merge=APS_mothur_Genus20.merge[,-1]
#APS_mothur_Genus20.merge1=as.data.frame(APS_mothur_Genus20.merge1)
#rownames(APS_mothur_Genus20.merge1)=APS_mothur_Genus20.merge$Genus
#colnames(APS_mothur_Genus20.merge1)=c("Sample-1")
#APS_mothur_Genus20.merge=APS_mothur_Genus20.merge1
#rm(APS_mothur_Genus20.merge1)
# anno=data.frame(factor(c(rep("A",25),rep("B",12),rep("C",6),rep("D",3))))
# rownames(anno)=colnames(APS_mothur_Genus20.merge)
# colnames(anno)="Group"
png("Top20_normalized_OTUs_Genus_heatmap.png", type="cairo", res = 200, units="in", width=8, height=4)
# pheatmap(APS_mothur_Genus20.merge,show_rownames = T,show_colnames = T,cluster_cols = F,scale = "none",annotation_col = anno,bGenus_color = F,fontsize_col = 7,fontsize_row = 8,fontsize = 7,cellheight = 10)
pheatmap(APS_mothur_Genus20.merge,show_rownames = T,show_colnames = T,cluster_cols = F,scale = "none",border_color = F,fontsize_col = 7,fontsize_row = 8,fontsize = 7,angle_col = 0)
dev.off()

In [ ]:
#Bar Plot
APS_mothur_Genus10 = data.frame(sort(taxa_sums(APS_mothur_Genus), TRUE)[1:10])
APS_mothur_Genus10$OTU=rownames(APS_mothur_Genus10)
APS_mothur_Genus10=APS_mothur_Genus10[,c(2,1)]
colnames(APS_mothur_Genus10)=c("OTU","Size")
APS_mothur_Genus10.merge=merge(APS_mothur_Genus10,Genus,by = "OTU",all.x = T)
APS_mothur_Genus10.merge=APS_mothur_Genus10.merge[rev(order(APS_mothur_Genus10.merge$Size)),]
as.matrix(names(APS_mothur_Genus10.merge))
APS_mothur_Genus10.merge=APS_mothur_Genus10.merge[,c(7:ncol(APS_mothur_Genus10.merge))]
APS_mothur_Genus10.merge
#APS_mothur_Genus10.merge=APS_mothur_Genus10.merge[,c(6:ncol(APS_mothur_Genus10.merge))]
#APS_mothur_Genus10.merge$Athul_MetG1 <- as.numeric(as.character(APS_mothur_Genus10.merge$Athul_MetG1))
#write.table(APS_mothur_Genus10.merge, file="Genus10.txt", sep = "\t",row.names = F,col.names = T, quote = F)
#APS_mothur_Genus10.merge=read.delim("Genus10.txt",header = T,stringsAsFactors = F, sep = "\t")
as.matrix(names(APS_mothur_Genus10.merge))
APS_mothur_Genus10.merge
colnames(APS_mothur_Genus10.merge)=c("Family","Genus",'CL', 'JS_F75', 'JS_R115', 'JS_V25', 'PM_F45', 'PM_R70', 'PM_V20', 'WD_F55', 'WD_R85', 'WD_V20')
cols.num <- names(APS_mothur_Genus10.merge[,c(3:ncol(APS_mothur_Genus10.merge))])
cols.num
APS_mothur_Genus10.merge[cols.num] <- sapply(APS_mothur_Genus10.merge[cols.num],as.numeric)

for( i in 2:length(APS_mothur_Genus10.merge))
{
  APS_mothur_Genus10.merge[,i]=round((APS_mothur_Genus10.merge[,i]/sum(APS_mothur_Genus10.merge[,i]))*100,digits = 1)
}
rm(i)
APS_mothur_Genus10.merge_melt=melt(APS_mothur_Genus10.merge)
APS_mothur_Genus10.merge
colnames(APS_mothur_Genus10.merge_melt)=c("Genus","Sample","Percentage")
png("Top10_normalized_OTUs_Genus_Barplot.png", type="cairo", res = 200, units="in", width=10, height=6)
#ggplot(top10Genus_barplot_melt,aes(x = Sample,y = Count,fill=Genus))+geom_bar(stat = "identity")
Genus_colors=rainbow(length(unique(APS_mothur_Genus10.merge$Genus)), s=.6, v=.9)[sample(1:length(unique(APS_mothur_Genus10.merge$Genus)),length(unique(APS_mothur_Genus10.merge$Genus)))]
ggplot(APS_mothur_Genus10.merge_melt,aes(x = Sample,y = Percentage,fill=Genus))+
  geom_bar(stat = "identity") + 
  scale_fill_manual(values = Genus_colors)+
  theme(axis.text.x = element_text(angle = 0, hjust = 1))
dev.off()


In [ ]:
#Bar Plot
APS_mothur_Genus10 = data.frame(sort(taxa_sums(APS_mothur_Genus), TRUE)[1:10])
APS_mothur_Genus10$OTU=rownames(APS_mothur_Genus10)
APS_mothur_Genus10=APS_mothur_Genus10[,c(2,1)]
colnames(APS_mothur_Genus10)=c("OTU","Size")
APS_mothur_Genus10.merge=merge(APS_mothur_Genus10,Genus,by = "OTU",all.x = T)
APS_mothur_Genus10.merge=APS_mothur_Genus10.merge[rev(order(APS_mothur_Genus10.merge$Size)),]
as.matrix(names(APS_mothur_Genus10.merge))
APS_mothur_Genus10.merge=APS_mothur_Genus10.merge[,c(7:ncol(APS_mothur_Genus10.merge))]
APS_mothur_Genus10.merge

In [ ]:
APS_mothur_Genus10.merge_new = paste(APS_mothur_Genus10.merge$Family, ';', APS_mothur_Genus10.merge$Genus)
as.matrix(APS_mothur_Genus10.merge_new)

In [ ]:
APS_mothur_Genus10.merge=merge(APS_mothur_Genus10.merge,APS_mothur_Genus10.merge_new)

In [ ]:
APS_mothur_Genus10.merge=a[,c(3:ncol(a))]

In [ ]:
APS_mothur_Genus10.merge

In [ ]:
as.matrix(names(APS_mothur_Genus10.merge))
colnames(APS_mothur_Genus10.merge)=c('Family','Genus','CL', 'JS_F75', 'JS_R115', 'JS_V25', 'PM_F45', 'PM_R70', 'PM_V20', 'WD_F55', 'WD_R85', 'WD_V20')
cols.num <- names(APS_mothur_Genus10.merge[,c(3:ncol(APS_mothur_Genus10.merge))])
APS_mothur_Genus10.merge[cols.num] <- sapply(APS_mothur_Genus10.merge[cols.num],as.numeric)

for( i in 2:length(APS_mothur_Genus10.merge))
{
  APS_mothur_Genus10.merge[,i]=round((APS_mothur_Genus10.merge[,i]/sum(APS_mothur_Genus10.merge[,i]))*100,digits = 1)
}
rm(i)
APS_mothur_Genus10.merge_melt=melt(APS_mothur_Genus10.merge)
APS_mothur_Genus10.merge
colnames(APS_mothur_Genus10.merge_melt)=c("Genus","Sample","Percentage")
png("Top10_normalized_OTUs_Genus_Barplot.png", type="cairo", res = 200, units="in", width=10, height=6)
#ggplot(top10Genus_barplot_melt,aes(x = Sample,y = Count,fill=Genus))+geom_bar(stat = "identity")
Genus_colors=rainbow(length(unique(APS_mothur_Genus10.merge$Genus)), s=.6, v=.9)[sample(1:length(unique(APS_mothur_Genus10.merge$Genus)),length(unique(APS_mothur_Genus10.merge$Genus)))]
ggplot(APS_mothur_Genus10.merge_melt,aes(x = Sample,y = Percentage,fill=Genus))+
  geom_bar(stat = "identity") + 
  scale_fill_manual(values = Genus_colors)+
  theme(axis.text.x = element_text(angle = 0, hjust = 1))
dev.off()


In [ ]:
#contig length distribution histogram
length_distribution=read.delim(file = "input.trim.contigs.good.unique.good.summary",header = T,sep = "\t",comment.char = "#",stringsAsFactors = F)
png("Contig_length_distribution_histogram.png", type="cairo", res = 200, units="in", width=6, height=5)
hist(length_distribution$nbases,breaks = 100,xlim = c(350,550),main="Contig length Distribution",xlab = "Contig Length (Bases)",ylab = "Frequency")
dev.off()

In [ ]:
##relative abundance at species and genus level
#Raw OTU
otus=read.delim(file = "CG_DN_500_mothur_OTU_Table.splitted.tab",header = T,sep = "\t",comment.char = "#",stringsAsFactors = F)
as.matrix(names(otus))
otus_norm=otus
for( i in 9:length(otus_norm))
{
  otus_norm[,i]=round((otus_norm[,i]/sum(otus_norm[,i]))*100,digits = 2)
}
write.table(otus_norm, file = "CG_DN_500_mothur_OTU_Table.splitted.relative-abundance.tab",sep = "\t",row.names = F,col.names = T,quote = F)



In [ ]:
sampleID

In [ ]:
df5.head()

# Extract OTU,size,kingdom and Genus

In [18]:
Genus=df5[['OTU','Size','Kingdom',"Family",'Genus', 'CL','JS_F75','JS_R115','JS_V25','PM_F45','PM_R70','PM_V20',
           'WD_F55','WD_R85','WD_V20' ]]

In [19]:
Genus.to_csv("Genus.CG_DN_500_mothur_OTU_Table.tab",sep="\t",index=False)

In [20]:
Phylum=df5[['OTU','Size','Kingdom','Phylum','CL','JS_F75','JS_R115','JS_V25','PM_F45','PM_R70','PM_V20',
           'WD_F55','WD_R85','WD_V20' ]]

In [21]:
Phylum.to_csv("Phylum.CG_DN_500_mothur_OTU_Table.tab",sep="\t",index=False)

In [24]:
Species=df5[['OTU','Size','Kingdom','Species','CL','JS_F75','JS_R115','JS_V25','PM_F45','PM_R70','PM_V20',
           'WD_F55','WD_R85','WD_V20' ]]

In [25]:
Species.to_csv("Species.CG_DN_500_mothur_OTU_Table.tab",sep="\t",index=False)

In [ ]:
#Genus
otus=read.delim(file = "Genus.CG_DN_500_mothur_OTU_Table.tab",header = T,sep = "\t",comment.char = "#",stringsAsFactors = F)
as.matrix(names(otus))
otus_norm=otus
for( i in 9:length(otus_norm))
{
  otus_norm[,i]=round((otus_norm[,i]/sum(otus_norm[,i]))*100,digits = 2)
}
write.table(otus_norm, file = "Genus.CG_DN_500_mothur_OTU_Table.relative-abundance.tab",sep = "\t",row.names = F,col.names = T,quote = F)


In [ ]:
#Species
#otus=read.delim(file = "Species.CG_DN_536_mothur_OTU_Table.tab",header = T,sep = "\t",comment.char = "#",stringsAsFactors = F)
#as.matrix(names(otus))
#otus_norm=otus
#for( i in 10:length(otus_norm))
#{
#  otus_norm[,i]=round((otus_norm[,i]/sum(otus_norm[,i]))*100,digits = 2)
#}
#write.table(otus_norm, file = "Species.CG_DN_536_mothur_OTU_Table.relative-abundance.tab",sep = "\t",row.names = F,col.names = T,quote = F)


In [ ]:
#Phylum
otus=read.delim(file = "Phylum.CG_DN_500_mothur_OTU_Table.tab",header = T,sep = "\t",comment.char = "#",stringsAsFactors = F)
as.matrix(names(otus))
otus_norm=otus
for( i in 5:length(otus_norm))
{
  otus_norm[,i]=round((otus_norm[,i]/sum(otus_norm[,i]))*100,digits = 2)
}
write.table(otus_norm, file = "Phylum.CG_DN_500_mothur_OTU_Table.relative-abundance.tab",sep = "\t",row.names = F,col.names = T,quote = F)


In [ ]:
help(png)

In [ ]:
new=read.delim(file = "Genus_level.txt",header = T,sep = "\t",comment.char = "#",stringsAsFactors = F)

In [ ]:
APS_mothur_Genus10.merge=new[,c(6:ncol(new))]

In [ ]:
APS_mothur_Genus10.merge

In [ ]:
A=as.matrix(paste(APS_mothur_Genus10.merge$Genus,";",APS_mothur_Genus10.merge$Family))

In [ ]:
write.table(A, file = "Genus_Family.csv",sep = "\t",row.names = F,col.names = T,quote = F)

In [ ]:
write.table(APS_mothur_Genus10.merge, file = "APS_mothur_Genus10.merge.csv",sep = "\t",row.names = F,col.names = T,quote = F)

In [ ]:
APS_mothur_Genus10.merge=read.delim(file = "APS_mothur_Genus10.merge.csv",header = T,sep = "\t",comment.char = "#",stringsAsFactors = F)

In [ ]:
as.matrix(names(APS_mothur_Genus10.merge))


In [ ]:
APS_mothur_Genus10.merge


In [ ]:
colnames(APS_mothur_Genus10.merge)=c("Genus",'CL', 'JS_F75', 'JS_R115', 'JS_V25', 'PM_F45', 'PM_R70', 'PM_V20', 'WD_F55', 'WD_R85', 'WD_V20')

In [ ]:
cols.num <- names(APS_mothur_Genus10.merge[,c(2:ncol(APS_mothur_Genus10.merge))])
cols.num

In [ ]:
APS_mothur_Genus10.merge[cols.num] <- sapply(APS_mothur_Genus10.merge[cols.num],as.numeric)


In [ ]:
for( i in 2:length(APS_mothur_Genus10.merge))
{
  APS_mothur_Genus10.merge[,i]=round((APS_mothur_Genus10.merge[,i]/sum(APS_mothur_Genus10.merge[,i]))*100,digits = 1)
}
rm(i)


In [ ]:
APS_mothur_Genus10.merge_melt=melt(APS_mothur_Genus10.merge)
APS_mothur_Genus10.merge
colnames(APS_mothur_Genus10.merge_melt)=c("Genus","Sample","Percentage")
png("Top10_normalized_OTUs_Genus_Barplot.png", type="cairo", res = 200, units="in", width=10, height=6)
#ggplot(top10Genus_barplot_melt,aes(x = Sample,y = Count,fill=Genus))+geom_bar(stat = "identity")
Genus_colors=rainbow(length(unique(APS_mothur_Genus10.merge$Genus)), s=.6, v=.9)[sample(1:length(unique(APS_mothur_Genus10.merge$Genus)),length(unique(APS_mothur_Genus10.merge$Genus)))]
ggplot(APS_mothur_Genus10.merge_melt,aes(x = Sample,y = Percentage,fill=Genus))+
  geom_bar(stat = "identity") + 
  scale_fill_manual(values = Genus_colors)+
  theme(axis.text.x = element_text(angle = 0, hjust = 1))
dev.off()


In [ ]:

colnames(APS_mothur_Genus10.merge)=c("Family","Genus",'CL', 'JS_F75', 'JS_R115', 'JS_V25', 'PM_F45', 'PM_R70', 'PM_V20', 'WD_F55', 'WD_R85', 'WD_V20')
cols.num <- names(APS_mothur_Genus10.merge[,c(3:ncol(APS_mothur_Genus10.merge))])
cols.num
APS_mothur_Genus10.merge[cols.num] <- sapply(APS_mothur_Genus10.merge[cols.num],as.numeric)

for( i in 2:length(APS_mothur_Genus10.merge))
{
  APS_mothur_Genus10.merge[,i]=round((APS_mothur_Genus10.merge[,i]/sum(APS_mothur_Genus10.merge[,i]))*100,digits = 1)
}
rm(i)
APS_mothur_Genus10.merge_melt=melt(APS_mothur_Genus10.merge)
APS_mothur_Genus10.merge
colnames(APS_mothur_Genus10.merge_melt)=c("Genus","Sample","Percentage")
png("Top10_normalized_OTUs_Genus_Barplot.png", type="cairo", res = 200, units="in", width=10, height=6)
#ggplot(top10Genus_barplot_melt,aes(x = Sample,y = Count,fill=Genus))+geom_bar(stat = "identity")
Genus_colors=rainbow(length(unique(APS_mothur_Genus10.merge$Genus)), s=.6, v=.9)[sample(1:length(unique(APS_mothur_Genus10.merge$Genus)),length(unique(APS_mothur_Genus10.merge$Genus)))]
ggplot(APS_mothur_Genus10.merge_melt,aes(x = Sample,y = Percentage,fill=Genus))+
  geom_bar(stat = "identity") + 
  scale_fill_manual(values = Genus_colors)+
  theme(axis.text.x = element_text(angle = 0, hjust = 1))
dev.off()


In [ ]:
APS_mothur_Genus10 = data.frame(sort(taxa_sums(APS_mothur_Genus), TRUE)[1:10])
APS_mothur_Genus10$OTU=rownames(APS_mothur_Genus10)
APS_mothur_Genus10=APS_mothur_Genus10[,c(2,1)]
colnames(APS_mothur_Genus10)=c("OTU","Size")
APS_mothur_Genus10.merge=merge(APS_mothur_Genus10,Genus,by = "OTU",all.x = T)
APS_mothur_Genus10.merge=APS_mothur_Genus10.merge[rev(order(APS_mothur_Genus10.merge$Size)),]
as.matrix(names(APS_mothur_Genus10.merge))
APS_mothur_Genus10.merge=APS_mothur_Genus10.merge[,c(7:ncol(APS_mothur_Genus10.merge))]
APS_mothur_Genus10.merge

In [1]:
write.table(APS_mothur_Genus10.merge, file = "APS_mothur_Genus10.merge.csv",sep = "\t",row.names = F,col.names = T,quote = F)

ERROR: Error in is.data.frame(x): object 'APS_mothur_Genus10.merge' not found


In [ ]:
a=paste(APS_mothur_Genus10.merge$Family,";",APS_mothur_Genus10.merge$Genus)

In [ ]:
write.table(a, file = "a.csv",sep = "\t",row.names = F,col.names = T,quote = F)

In [ ]:
pwd